# Phase 6 — Model Development and Comparison

We train supervised regressors on the offline Kaggle-derived dataset. The target is a transparent proxy constructed from product rating, log review count, and discount percentage. It is a teaching target for regression and model comparison, not a user-labelled measure of personal relevance. Product-title groups are split so exact repeated titles cannot cross partitions. We use a development set for model selection and reserve a separate holdout set for Phase 7.

In [2]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

sys.path.insert(0, "..")
from backend.product_features import FEATURE_COLUMNS, product_feature_frame

DATA_PATH = Path("electronics_feature_engineered.csv")
ARTIFACT_DIR = Path("../backend/model_artifacts")
OUTPUT_DIR = Path("model_outputs")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
df = pd.read_csv(DATA_PATH)
print("Rows:", len(df), "| Feature count:", len(FEATURE_COLUMNS))

Rows: 5010 | Feature count: 16


## Target definition and split

For each product, rating quality is `rating / 5`; popularity is `log1p(review_count)` scaled by the largest observed log count; discount is clipped to 0–100% and scaled to 0–1. The weighted proxy is `0.45 × rating quality + 0.35 × popularity + 0.20 × discount`. Missing ratings use the training-data median for target construction, missing review counts are treated as zero, and missing discount information receives zero discount contribution. These choices keep all records and are stated as assumptions.

In [4]:
rating = pd.to_numeric(df["review_rating"], errors="coerce")
rating_fill = float(rating.median())
rating_quality = rating.fillna(rating_fill).clip(0, 5) / 5
df["no_of_ratings"] = pd.to_numeric(df["no_of_ratings"], errors="coerce").fillna(0).clip(lower=0)
reviews = df["no_of_ratings"]
log_reviews = np.log1p(reviews)
log_review_max = float(log_reviews.max()) or 1.0
popularity = log_reviews / log_review_max
discount = pd.to_numeric(df["discount_percent"], errors="coerce").clip(0, 100).fillna(0) / 100

y = (0.45 * rating_quality + 0.35 * popularity + 0.20 * discount).astype(float)
X = product_feature_frame(df)
print("Target range:", round(float(y.min()), 4), "to", round(float(y.max()), 4))
print("Missing target values:", int(y.isna().sum()))
print("Predictor columns in fixed order:", FEATURE_COLUMNS)

groups = df["name"].fillna("").astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
# First reserve a final holdout. It is not used to choose the model.
holdout_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
dev_idx, test_idx = next(holdout_splitter.split(X, y, groups=groups))
X_dev, X_test = X.iloc[dev_idx], X.iloc[test_idx]
y_dev, y_test = y.iloc[dev_idx], y.iloc[test_idx]
groups_dev = groups.iloc[dev_idx]
# Use a grouped validation subset within development data for model selection.
validation_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=43)
train_rel, val_rel = next(validation_splitter.split(X_dev, y_dev, groups=groups_dev))
X_train, X_val = X_dev.iloc[train_rel], X_dev.iloc[val_rel]
y_train, y_val = y_dev.iloc[train_rel], y_dev.iloc[val_rel]
print("Model-selection train rows:", len(X_train), "| Validation rows:", len(X_val))
print("Development rows for final fit:", len(X_dev), "| Reserved holdout rows:", len(X_test))
print("Exact product-title groups overlap across holdout:", bool(set(groups_dev) & set(groups.iloc[test_idx])))
print("Exact product-title groups overlap across train/validation:", bool(set(groups_dev.iloc[train_rel]) & set(groups_dev.iloc[val_rel])))

Target range: 0.1718 to 0.9501
Missing target values: 0
Predictor columns in fixed order: ['price_log1p', 'rating', 'review_count_log1p', 'discount_percent', 'gaming', 'wireless', 'rgb', 'battery_present', 'premium_brand', 'fast_charging', 'ram_gb', 'storage_gb', 'battery_mah', 'screen_inches', 'charging_w', 'refresh_rate_hz']
Model-selection train rows: 3185 | Validation rows: 804
Development rows for final fit: 3989 | Reserved holdout rows: 1021
Exact product-title groups overlap across holdout: False
Exact product-title groups overlap across train/validation: False


## Baseline and candidate regressors

The mean baseline predicts the development-training target mean for every validation product. Linear Regression provides an interpretable reference, while a Decision Tree and Random Forest can model nonlinear patterns. Median imputation is fitted only on each training partition inside the pipeline. Candidates are compared on a grouped validation split. The separate holdout remains unused until Phase 7. MAE and RMSE are lower-is-better; R² compares the model with the mean baseline.

In [6]:
models = {
    "Linear Regression": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", LinearRegression())]),
    "Decision Tree": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", DecisionTreeRegressor(max_depth=8, min_samples_leaf=10, random_state=42))]),
    "Random Forest": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features=0.8, random_state=42, n_jobs=-1))]),
}

def metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred), "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)), "R2": r2_score(y_true, y_pred)}

baseline_pred = np.full(len(y_val), float(y_train.mean()))
results = [{"Model": "Mean Baseline", **metrics(y_val, baseline_pred)}]
for name, model in models.items():
    model.fit(X_train, y_train)
    results.append({"Model": name, **metrics(y_val, model.predict(X_val))})
results_df = pd.DataFrame(results).sort_values(["RMSE", "MAE"]).reset_index(drop=True)
display(results_df.round(4))
results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

candidate = results_df.loc[results_df.Model != "Mean Baseline"].iloc[0]["Model"]
selection_model = models[candidate]
validation_predictions = selection_model.predict(X_val)
print("Selected by lowest validation RMSE:", candidate)
# After selecting hyperparameters/model family, refit on all development rows.
final_model = models[candidate]
final_model.fit(X_dev, y_dev)

            Model    MAE   RMSE      R2
    Random Forest 0.0053 0.0122  0.9902
    Decision Tree 0.0153 0.0213  0.9702
Linear Regression 0.0111 0.0241  0.9620
    Mean Baseline 0.0921 0.1234 -0.0006
Selected by lowest validation RMSE: Random Forest


In [7]:
# Compare model errors and inspect actual versus predicted values for the selected model.
selected_predictions = validation_predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(results_df.Model, results_df.RMSE, color=["#9aa4b2" if name == "Mean Baseline" else "#2f6f8f" for name in results_df.Model])
axes[0].set_ylabel("RMSE")
axes[0].set_title("Validation Error by Model")
axes[0].tick_params(axis="x", rotation=25)
axes[1].scatter(y_val, selected_predictions, alpha=0.35, s=18)
axes[1].plot([0, 1], [0, 1], color="black", linestyle="--")
axes[1].set(xlabel="Actual proxy score", ylabel="Predicted proxy score", title=f"Actual vs predicted: {candidate}")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "phase6_model_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

# Save the exact estimator, input order and target assumptions used for inference.
artifact = {
    "model": final_model,
    "model_name": candidate,
    "feature_columns": FEATURE_COLUMNS,
    "target_definition": "0.45*(rating/5) + 0.35*(log1p(review_count)/training_max_log1p_review_count) + 0.20*(clipped_discount_percent/100)",
    "target_rating_fill": rating_fill,
    "target_log_review_max": log_review_max,
    "random_state": 42,
    "test_size": 0.20,
    "validation_size": 0.20,
    "split_method": "GroupShuffleSplit by normalized exact product title, random states 42 and 43",
    "metrics": results_df.to_dict(orient="records"),
}
joblib.dump(artifact, ARTIFACT_DIR / "product_score.joblib")
print("Saved model artifact:", ARTIFACT_DIR / "product_score.joblib")

06_model_development.ipynb:cell6:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Saved model artifact: ../backend/model_artifacts/product_score.joblib


In [8]:
# Feature importance describes contribution to the constructed proxy, not causal importance.
estimator = final_model.named_steps["model"]
if hasattr(estimator, "feature_importances_"):
    importance_values = estimator.feature_importances_
elif hasattr(estimator, "coef_"):
    importance_values = np.abs(estimator.coef_)
else:
    importance_values = np.zeros(len(FEATURE_COLUMNS))
importance = pd.DataFrame({"feature": FEATURE_COLUMNS, "importance": importance_values}).sort_values("importance", ascending=False)
display(importance.head(12).round(4))
importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 5))
top = importance.head(12).sort_values("importance")
ax.barh(top.feature, top.importance, color="#2f6f8f")
ax.set(title="Top model feature importances", xlabel="Importance", ylabel="Feature")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "phase6_feature_importance.png", dpi=180, bbox_inches="tight")
plt.show()

           feature  importance
            rating      0.6075
review_count_log1p      0.2418
  discount_percent      0.1447
       price_log1p      0.0051
          wireless      0.0003
     premium_brand      0.0002
            ram_gb      0.0001
     screen_inches      0.0001
        storage_gb      0.0001
   battery_present      0.0001
     fast_charging      0.0001
        charging_w      0.0000
06_model_development.ipynb:cell7:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Interpretation

The chosen regressor estimates the constructed quality/popularity/discount proxy. Several target components also appear among predictors, so strong holdout metrics show that the model reproduces this proxy on a random split. They do not establish personalization, causal effects, or recommendation relevance. Phase 7 checks residuals and train/test differences, then the live ranking layer combines the proxy estimate with explicit query constraints.